In [2]:
import os
import re
import PyPDF2

def extract_aadhar_details(pdf_path):
    """
    Extracts Aadhar card number and name from a PDF file.

    Args:
        pdf_path: Path to the PDF file.

    Returns:
        A dictionary containing the Aadhar number and name, or None if 
        extraction fails.  Returns an empty dictionary if the PDF is not readable.
    """
    try:
        with open(pdf_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)

            num_pages = len(reader.pages)
            text = ""
            for i in range(num_pages):
                page = reader.pages[i]
                text += page.extract_text()


        aadhar_number = None
        name = None

        # Regex patterns (improve these based on your Aadhar card PDFs)
        aadhar_pattern = r"\b\d{4}\s?\d{4}\s?\d{4}\b"  # Matches 12-digit Aadhar number
        name_pattern = r"Name\s*:\s*(.*?)\n"  # Matches "Name: " followed by the name
        #name_pattern = r"Name\s*:\s*(.*?)(?:\n|$)" #Another name pattern to try

        print(text)
        aadhar_match = re.search(aadhar_pattern, text)
        if aadhar_match:
            aadhar_number = aadhar_match.group(0).replace(" ", "")  # Remove spaces

        name_match = re.search(name_pattern, text, re.IGNORECASE) #Case-insensitive search
        if name_match:
            name = name_match.group(1).strip()  # Remove leading/trailing spaces


        if aadhar_number and name:
            return {"aadhar_number": aadhar_number, "name": name}
        else:
          return None #Return None if either aadhar or name is not found


    except FileNotFoundError:
        print(f"Error: File not found: {pdf_path}")
        return None
    except PyPDF2.errors.PdfReadError:
        print(f"Error: Could not read PDF: {pdf_path}. It might be corrupted or not a valid PDF.")
        return {} #Empty dict for unreadable PDFs
    except Exception as e: # Catch other potential errors
        print(f"An unexpected error occurred processing {pdf_path}: {e}")
        return None



def process_aadhar_pdfs(directory):
    """
    Processes all Aadhar card PDFs in a directory.

    Args:
        directory: Path to the directory containing the PDFs.
    """

    for filename in os.listdir(directory):
        if filename.endswith(".pdf"):
            pdf_path = os.path.join(directory, filename)
            extracted_data = extract_aadhar_details(pdf_path)

            if extracted_data:
                print(f"File: {filename}")
                print(f"  Aadhar Number: {extracted_data['aadhar_number']}")
                print(f"  Name: {extracted_data['name']}")
                print("-" * 20)
            elif extracted_data is None:
                print(f"Could not extract data from: {filename}")
                print("-" * 20)
            elif not extracted_data: # Empty dict means PDF read error
                print(f"Error reading PDF: {filename}")
                print("-" * 20)


# Example usage:
directory_path = r"C:\Nandeesh\my_docs\Nandeesh_adhar_card.pdf"  # Replace with your directory path
# process_aadhar_pdfs(directory_path)
x = extract_aadhar_details(directory_path)
x

Scanned with CamScanner
Scanned with CamScanner



In [5]:
import os
import re
import pandas as pd
import numpy as np
from paddleocr import PaddleOCR
import cv2  # Import OpenCV


# Initialize PaddleOCR (do this only once)
ocr = PaddleOCR(use_angle_cls=True, lang='en')  # Set lang='en' for English

def extract_aadhar_details_paddle(pdf_path):
    """
    Extracts Aadhar card number and name from a PDF file using PaddleOCR.

    Args:
        pdf_path: Path to the PDF file.

    Returns:
        A dictionary containing the Aadhar number and name, or None if 
        extraction fails.
    """
    try:
        from PIL import Image
        import fitz  # PyMuPDF

        doc = fitz.open(pdf_path)
        extracted_data = []
        print("entering")
        for page_num in range(len(doc)):
            page = doc[page_num]
            pix = page.get_pixmap(matrix=fitz.Matrix(3, 3)) # Increase resolution for better OCR
            # img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

            # ... inside the loop ...
            # pix = page.get_pixmap(matrix=fitz.Matrix(3, 3))
            img_cv = cv2.imdecode(np.frombuffer(pix.samples, np.uint8), cv2.IMREAD_COLOR) #Using cv2
            img = Image.fromarray(img_cv) #Converting into PIL image
            
            result = ocr.ocr(img)
            print(result)

            page_text = ""
            for line in result:
                page_text += " ".join([word[0] for word in line[1]]) + "\n" #Join words in each line and then lines with newline

            extracted_data.append(page_text)
        
        full_text = "".join(extracted_data) #Combine all the page texts
        print(f"full_text is {full_text}")

        aadhar_number = None
        name = None

        aadhar_pattern = r"\b\d{4}\s?\d{4}\s?\d{4}\b"  # Matches 12-digit Aadhar number
        name_pattern = r"Name\s*:\s*(.*?)\n"  # Matches "Name: " followed by the name
        #name_pattern = r"Name\s*:\s*(.*?)(?:\n|$)" #Another name pattern to try


        aadhar_match = re.search(aadhar_pattern, full_text)
        if aadhar_match:
            aadhar_number = aadhar_match.group(0).replace(" ", "")  # Remove spaces

        name_match = re.search(name_pattern, full_text, re.IGNORECASE) #Case-insensitive search
        if name_match:
            name = name_match.group(1).strip()  # Remove leading/trailing spaces


        if aadhar_number and name:
            return {"aadhar_number": aadhar_number, "name": name}
        else:
          return None #Return None if either aadhar or name is not found

    except Exception as e:
        print(f"An error occurred processing {pdf_path}: {e}")
        print(f"error {e}")
        return None


def process_aadhar_pdfs(directory, output_excel):
    """
    Processes all Aadhar card PDFs in a directory and saves the results to an Excel file.

    Args:
        directory: Path to the directory containing the PDFs.
        output_excel: Path to the output Excel file.
    """

    data = []
    for filename in os.listdir(directory):
        if filename.endswith(".pdf"):
            pdf_path = os.path.join(directory, filename)
            extracted_data = extract_aadhar_details_paddle(pdf_path)

            if extracted_data:
                data.append({"filename": filename, **extracted_data})
            else:
                data.append({"filename": filename, "aadhar_number": None, "name": None})  # Store None if extraction fails

    df = pd.DataFrame(data)
    df.to_excel(output_excel, index=False)
    print(f"Results saved to {output_excel}")



# Example usage:
directory_path = r"C:\Nandeesh\my_docs\Nandeesh_adhar_card.pdf"  # Replace with your directory path
output_excel_file = "aadhar_data.xlsx"  # Name of your output Excel file
# process_aadhar_pdfs(directory_path, output_excel_file)
x = extract_aadhar_details_paddle(directory_path)
x

[2025/02/11 23:53:49] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\nande/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\nande/.paddleocr/whl\\rec\\en\\en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=

In [4]:
directory_path = r"C:\Nandeesh\my_docs\Nandeesh_adhar_card.pdf"  # Replace with your directory path
output_excel_file = "aadhar_data.xlsx"  # Name of your output Excel file
# process_aadhar_pdfs(directory_path, output_excel_file)
x = extract_aadhar_details_paddle(directory_path)
x

Error processing C:\Nandeesh\my_docs\Nandeesh_adhar_card.pdf: 


In [ ]:
import cv2  # Import OpenCV

# ... inside the loop ...
pix = page.get_pixmap(matrix=fitz.Matrix(3, 3))
img_cv = cv2.imdecode(np.frombuffer(pix.samples, np.uint8), cv2.IMREAD_COLOR) #Using cv2
img = Image.fromarray(img_cv) #Converting into PIL image
# ... rest of your code ...

In [8]:
x = ocr.ocr("image.png")
x

[2025/02/11 23:41:05] ppocr DEBUG: dt_boxes num : 34, elapsed : 0.45103001594543457
[2025/02/11 23:41:05] ppocr DEBUG: cls num  : 34, elapsed : 0.1450355052947998
[2025/02/11 23:41:08] ppocr DEBUG: rec_res num  : 34, elapsed : 2.7250514030456543


[[[[[1078.0, 239.0], [1254.0, 239.0], [1254.0, 276.0], [1078.0, 276.0]],
   ('STrT', 0.5010800957679749)],
  [[[420.0, 463.0], [1333.0, 460.0], [1333.0, 497.0], [420.0, 500.0]],
   ('Lniaue identifieation Autbority ofindis', 0.8744587302207947)],
  [[[590.0, 518.0], [1179.0, 516.0], [1179.0, 571.0], [590.0, 574.0]],
   ('Government of India', 0.9564610719680786)],
  [[[473.0, 595.0], [1288.0, 589.0], [1288.0, 631.0], [473.0, 637.0]],
   ('cowoa/Enrolment No.: 2738/10617/18544', 0.8875203728675842)],
  [[[560.0, 658.0], [600.0, 658.0], [600.0, 684.0], [560.0, 684.0]],
   ('To', 0.9210291504859924)],
  [[[560.0, 713.0], [746.0, 713.0], [746.0, 742.0], [560.0, 742.0]],
   ('Nandeesh KM', 0.9456180334091187)],
  [[[558.0, 739.0], [869.0, 739.0], [869.0, 774.0], [558.0, 774.0]],
   ('S/O:Chidanandalah K M', 0.9403344988822937)],
  [[[558.0, 768.0], [739.0, 774.0], [738.0, 803.0], [557.0, 797.0]],
   ('kvor colony', 0.9569284915924072)],
  [[[558.0, 797.0], [831.0, 797.0], [831.0, 831.0], [5

In [3]:
import os
import re
import pandas as pd
from paddleocr import PaddleOCR
import fitz  # PyMuPDF
from PIL import Image

# Initialize PaddleOCR (do this only once)
ocr = PaddleOCR(use_angle_cls=True, lang='en')  # Set lang='en' for English

def extract_aadhar_details_paddle(pdf_path):
    """Extracts Aadhar card details using PaddleOCR."""
    try:
        doc = fitz.open(pdf_path)
        extracted_data = []

        for page_num in range(len(doc)):
            page = doc[page_num]
            pix = page.get_pixmap(matrix=fitz.Matrix(3, 3))  # Increased resolution
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

            result = ocr.ocr(img)

            page_text = ""
            for line in result:
                page_text += " ".join([word[0] for word in line[1]]) + "\n"

            extracted_data.append(page_text)

        full_text = "".join(extracted_data)

        aadhar_number = None
        name = None

        # More robust Aadhar number regex (handles variations)
        aadhar_pattern = r"\b\d{4}\s?\d{4}\s?\d{4}\b|\b\d{12}\b"  # 12 digits with or without spaces

        # More flexible name regex (handles "Name:", "NAME:", etc.)
        name_pattern = r"(?:Name|NAME|Name:|NAME:)\s*:\s*(.*?)(?:\n|$)"

        aadhar_match = re.search(aadhar_pattern, full_text)
        if aadhar_match:
            aadhar_number = aadhar_match.group(0).replace(" ", "")

        name_match = re.search(name_pattern, full_text, re.IGNORECASE)
        if name_match:
            name = name_match.group(1).strip()

        return {"aadhar_number": aadhar_number, "name": name} if aadhar_number and name else None

    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")
        return None


def process_aadhar_pdfs(directory, output_excel):
    """Processes PDFs and saves to Excel."""
    data = []
    for filename in os.listdir(directory):
        if filename.endswith(".pdf"):
            pdf_path = os.path.join(directory, filename)
            extracted_data = extract_aadhar_details_paddle(pdf_path)
            data.append({"filename": filename, **(extracted_data or {"aadhar_number": None, "name": None})})

    df = pd.DataFrame(data)
    df.to_excel(output_excel, index=False)
    print(f"Results saved to {output_excel}")


# Example usage:
directory_path = "/path/to/your/aadhar/pdfs"  # Replace with your path
output_excel_file = "aadhar_data.xlsx"
process_aadhar_pdfs(directory_path, output_excel_file)

[2025/02/11 23:37:07] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\nande/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\nande/.paddleocr/whl\\rec\\en\\en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/path/to/your/aadhar/pdfs'

In [ ]:
l = [1,2,3,9]
# sum = 10 and 8


In [ ]:
C:\Nandeesh\git\kn_ml_course\LSTM RNN